# Class 03 — Lab: Classification Metrics II
# Aula 03 — Laboratório: Métricas de Classificação II

## Topics Studied:
*   ROC (Receiver Operating Characteristic) Curve
*   AUC (Area Under the ROC Curve)
*   Precision-Recall Curve
*   AP (Average Precision)
*   Log-loss
*   Threshold Selection
*   Multiclass Metrics (Macro, Micro, Weighted Averages)
*   Model Comparison with Baselines

## Tópicos Estudados:
*   Curva ROC (Receiver Operating Characteristic)
*   AUC (Área Sob a Curva ROC)
*   Curva Precisão-Recall
*   AP (Average Precision)
*   Log-loss
*   Seleção de Limiar
*   Métricas Multiclasse (Médias Macro, Micro, Ponderada)
*   Comparação de Modelos com Baselines

# Class 03 — Lab: Classification Metrics II

**ROC, AUC, Precision-Recall, AP, and log-loss.** We will train a classifier and evaluate it from all angles, connecting with Class 1 (data cleaning) and Class 2 (confusion matrix, precision, recall, and threshold).

**Datasets:** `datasets/5_musicas.csv` (predict `hit`, rare positive ~12%) and `datasets/1_pacientes.csv` (multi-class `risco_metabolico`).

# Aula 03 — Laboratório: Métricas de classificação II

**ROC, AUC, Precisão-Recall, AP e log-loss.** Vamos treinar um classificador e avaliá-lo por todos os ângulos, ligando com a Aula 1 (limpeza) e a Aula 2 (matriz de confusão, precisão, recall e limiar).

**Bases:** `datasets/5_musicas.csv` (prever `hit`, positivo raro ~12%) e `datasets/1_pacientes.csv` (multiclasse `risco_metabolico`).

## Part 0 — Environment, data, and a model
## Parte 0 — Ambiente, dados e um modelo

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

mus = pd.read_csv("/content/5_musicas.csv")
# deixamos 'popularidade' de fora de propósito: ela quase entrega o hit
# (o modelo ficaria 'perfeito' e as métricas perderiam a nuance).
feat = ["danceability","energy","valence","acousticness","instrumentalness",
        "speechiness","liveness","loudness","tempo","duration_ms"]
X, y = mus[feat], mus["hit"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
proba = modelo.predict_proba(X_test)[:, 1]        # a NOTA (probabilidade) de ser hit
print("treino:", X_train.shape, "| positivos (hit) no teste:", int(y_test.sum()), "de", len(y_test))

treino: (735, 10) | positivos (hit) no teste: 38 de 315


## Part 1 — Cleaning (Class 1) and the threshold (Class 2)
## Parte 1 — Limpeza (Aula 1) e o limiar (Aula 2)

**Exercise 1 (Class 1).** Before trusting the model, check the `mus` dataset: how many missing values are there per column and how many duplicate rows? (`isna().sum()` and `duplicated().sum()`).
**Exercício 1 (Aula 1).** Antes de confiar no modelo, verifique a base `mus`: quantos valores faltantes há por coluna e quantas linhas duplicadas? (`isna().sum()` e `duplicated().sum()`).

In [ ]:
print("Valores faltantes por coluna:\n", mus.isna().sum())
print("\nNúmero de linhas duplicadas:", mus.duplicated().sum())

Valores faltantes por coluna:
 id                  0
titulo              0
artista             0
ano                 0
genero              0
danceability        0
energy              0
valence             0
acousticness        0
instrumentalness    0
speechiness         0
liveness            0
loudness            0
tempo               0
duration_ms         0
popularidade        0
hit                 0
dtype: int64

Número de linhas duplicadas: 0


**Exercise 2 (Class 2).** At the default threshold of 0.5, generate predictions and show the **confusion matrix**, **precision**, and **recall`.
**Exercício 2 (Aula 2).** No limiar padrão 0,5, gere as previsões e mostre a **matriz de confusão** e a **precisão** e o **recall`.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

y_pred = (proba >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Matriz de Confusão:\n", cm)
print(f"Precisão: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Matriz de Confusão:
 [[272   5]
 [ 23  15]]
Precisão: 0.7500
Recall: 0.3947


**Exercise 3 (the question they got wrong).** Recalculate precision and recall at thresholds **0.3** and **0.7**. Confirm: lowering the threshold **increases recall** and **reduces precision`.
**Exercício 3 (a questão que erraram).** Recalcule precisão e recall nos limiares **0,3** e **0,7**. Confirme: baixar o limiar **aumenta o recall** e **reduz a precisão`.

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Limiar 0.3
y_pred_03 = (proba >= 0.3).astype(int)
precision_03 = precision_score(y_test, y_pred_03)
recall_03 = recall_score(y_test, y_pred_03)
print(f"Limiar 0.3: Precisão = {precision_03:.4f}, Recall = {recall_03:.4f}")

# Limiar 0.7
y_pred_07 = (proba >= 0.7).astype(int)
precision_07 = precision_score(y_test, y_pred_07)
recall_07 = recall_score(y_test, y_pred_07)
print(f"Limiar 0.7: Precisão = {precision_07:.4f}, Recall = {recall_07:.4f}")

print("\nConfirmação: baixar o limiar (0.5 para 0.3) aumenta o recall e reduz a precisão.")
print("              aumentar o limiar (0.5 para 0.7) reduz o recall e aumenta a precisão (ou mantém, se não houver predições).")

Limiar 0.3: Precisão = 0.6341, Recall = 0.6842
Limiar 0.7: Precisão = 0.8889, Recall = 0.2105

Confirmação: baixar o limiar (0.5 para 0.3) aumenta o recall e reduz a precisão.
              aumentar o limiar (0.5 para 0.7) reduz o recall e aumenta a precisão (ou mantém, se não houver predições).


## Part 2 — ROC Curve and AUC
## Parte 2 — Curva ROC e AUC

**Exercise 4.** Calculate the **AUC** with `roc_auc_score(y_test, proba)` and interpret the value (0.5 = chance; 1.0 = perfect).
**Exercício 4.** Calcule a **AUC** com `roc_auc_score(y_test, proba)` e interprete o valor (0,5 = acaso; 1,0 = perfeito).

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, proba)
print(f"AUC: {auc:.4f}")

if auc > 0.9:
    interpretation = "excelente"
elif auc > 0.8:
    interpretation = "muito bom"
elif auc > 0.7:
    interpretation = "bom"
elif auc > 0.6:
    interpretation = "razoável"
elif auc >= 0.5:
    interpretation = "fraco (pouco melhor que o acaso)"
else:
    interpretation = "pior que o acaso"

print(f"Interpretação da AUC: O modelo tem uma capacidade de distinção {interpretation}.")

AUC: 0.9147
Interpretação da AUC: O modelo tem uma capacidade de distinção excelente.


**Exercise 5.** Generate `roc_curve(y_test, proba)` and show some points (fpr, tpr). What happens to the false alarm (fpr) as recall (tpr) increases?
**Exercício 5.** Gere `roc_curve(y_test, proba)` e mostre alguns pontos (fpr, tpr). O que acontece com o falso alarme (fpr) conforme o recall (tpr) sobe?

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, proba)

# Mostra alguns pontos da curva ROC
print("Alguns pontos da curva ROC (FPR, TPR, Limiar):\n")
# Display points at roughly 0.2, 0.4, 0.6, 0.8, 0.95 TPR
indices = np.searchsorted(tpr, [0.2, 0.4, 0.6, 0.8, 0.95])
for i in indices:
    print(f"FPR: {fpr[i]:.4f}, TPR (Recall): {tpr[i]:.4f}, Limiar: {thresholds[i]:.4f}")

print("\nObservação: Conforme o recall (TPR) sobe, o falso alarme (FPR) também tende a subir.")

Alguns pontos da curva ROC (FPR, TPR, Limiar):

FPR: 0.0036, TPR (Recall): 0.2368, Limiar: 0.6982
FPR: 0.0181, TPR (Recall): 0.4211, Limiar: 0.4987
FPR: 0.0361, TPR (Recall): 0.6053, Limiar: 0.3812
FPR: 0.1300, TPR (Recall): 0.8421, Limiar: 0.1723
FPR: 0.4368, TPR (Recall): 0.9737, Limiar: 0.0279

Observação: Conforme o recall (TPR) sobe, o falso alarme (FPR) também tende a subir.


**Exercise 6.** Confirm in practice the meaning of AUC: draw **1000 pairs** (one hit and one non-hit) and calculate the fraction in which the hit received a higher score. It should match the AUC.
**Exercício 6.** Confirme na prática o significado da AUC: sorteie **1000 pares** (um hit e um não-hit) e calcule a fração em que o hit recebeu nota maior. Deve bater com a AUC.

In [ ]:
np.random.seed(42) # Para reprodutibilidade

# Pega as probabilidades para hits (y_test=1) e non-hits (y_test=0)
proba_hits = proba[y_test == 1]
proba_non_hits = proba[y_test == 0]

num_simulations = 1000
hits_higher = 0

for _ in range(num_simulations):
    # Sorteia um hit e um non-hit aleatoriamente
    hit_score = np.random.choice(proba_hits)
    non_hit_score = np.random.choice(proba_non_hits)

    if hit_score > non_hit_score:
        hits_higher += 1

fraction_hits_higher = hits_higher / num_simulations

print(f"Fração de vezes que a probabilidade do hit foi maior que a do non-hit (simulação): {fraction_hits_higher:.4f}")
print(f"AUC calculada anteriormente: {auc:.4f}")
print("Os valores devem ser próximos, confirmando o significado da AUC.")

Fração de vezes que a probabilidade do hit foi maior que a do non-hit (simulação): 0.9220
AUC calculada anteriormente: 0.9147
Os valores devem ser próximos, confirmando o significado da AUC.


## Part 3 — Precision-Recall and Average Precision (AP)
## Parte 3 — Precisão-Recall e average precision (AP)

🇬🇧  
**Exercise 7.** Calculate the **AP** with `average_precision_score(y_test, proba)` and compare with the positive rate (the baseline of a random guessing model).

🇧🇷
**Exercício 7.** Calcule a **AP** com `average_precision_score(y_test, proba)` e compare com a taxa de positivos (a linha de base de um modelo que chuta ao acaso).

In [ ]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_test, proba)
print(f"Average Precision (AP): {ap:.4f}")

# Taxa de positivos (baseline)
positive_rate = y_test.sum() / len(y_test)
print(f"Taxa de positivos (baseline): {positive_rate:.4f}")

print("\nComparação: A AP deve ser significativamente maior que a taxa de positivos, indicando que o modelo tem poder preditivo real sobre a classe positiva rara.")

Average Precision (AP): 0.6698
Taxa de positivos (baseline): 0.1206

Comparação: A AP deve ser significativamente maior que a taxa de positivos, indicando que o modelo tem poder preditivo real sobre a classe positiva rara.


**Exercise 8.** Generate `precision_recall_curve` and show the precision at three recall levels (~0.5, ~0.8, ~0.95). Does precision decrease as recall increases?
**Exercício 8.** Gere `precision_recall_curve` e mostre a precisão em três níveis de recall (~0,5, ~0,8, ~0,95). A precisão cai conforme o recall sobe?

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, proba)

print("Precisão em diferentes níveis de Recall (aproximados):")

# Encontrar índices para níveis de recall desejados (0.5, 0.8, 0.95)
# Usamos np.argmin(np.abs(recall - target_recall)) para encontrar o recall mais próximo

recall_levels = [0.5, 0.8, 0.95]
for target_recall in recall_levels:
    # Encontra o índice onde o recall está mais próximo do target_recall
    idx = np.argmin(np.abs(recall - target_recall))
    print(f"  Recall mais próximo de {target_recall:.2f}: {recall[idx]:.4f}, Precisão: {precision[idx]:.4f}")

print("\nObservação: A precisão geralmente cai à medida que o recall sobe, pois o modelo precisa fazer mais previsões positivas para capturar mais verdadeiros positivos, o que pode aumentar os falsos positivos e, consequentemente, diminuir a precisão.")

Precisão em diferentes níveis de Recall (aproximados):
  Recall mais próximo de 0.50: 0.5000, Precisão: 0.6786
  Recall mais próximo de 0.80: 0.7895, Precisão: 0.4545
  Recall mais próximo de 0.95: 0.9474, Precisão: 0.2293

Observação: A precisão geralmente cai à medida que o recall sobe, pois o modelo precisa fazer mais previsões positivas para capturar mais verdadeiros positivos, o que pode aumentar os falsos positivos e, consequentemente, diminuir a precisão.


We can clearly observe that as recall increases, precision tends to decrease, which is expected behavior in Precision-Recall curves. This happens because, to capture more true positives (increase recall), the model needs to be less selective, which inevitably leads to an increase in false positives and, consequently, a drop in precision.
udemos observar claramente que, à medida que o recall aumenta, a precisão tende a diminuir, o que é um comportamento esperado nas curvas de Precisão-Recall. Isso acontece porque, para capturar mais verdadeiros positivos (aumentar o recall), o modelo precisa ser menos seletivo, o que inevitavelmente leva a um aumento de falsos positivos e, consequentemente, à queda da precisão.

**Exercise 9 (ROC vs PR in imbalanced data).** Make the positive class even rarer: use only half of the test hits. Recalculate AUC and AP. Which of the two drops more? (AP is more sensitive to rare positives).
**Exercício 9 (ROC x PR no desbalanceamento).** Torne o positivo ainda mais raro: use só metade dos hits do teste. Recalcule AUC e AP. Qual das duas cai mais? (a AP é mais sensível ao positivo raro).

In [ ]:
# Copia os y_test e proba originais
y_test_original = y_test.copy()
proba_original = proba.copy()

# Identifica os índices dos hits (classe positiva)
hit_indices = np.where(y_test_original == 1)[0]

# Seleciona apenas metade dos hits
num_hits_to_keep = len(hit_indices) // 2
np.random.seed(42) # Para reprodutibilidade
selected_hit_indices = np.random.choice(hit_indices, num_hits_to_keep, replace=False)

# Identifica os índices dos não-hits
non_hit_indices = np.where(y_test_original == 0)[0]

# Combina os índices selecionados
new_indices = np.concatenate([selected_hit_indices, non_hit_indices])

# Cria os novos y_test e proba com a classe positiva mais rara
y_test_rare = y_test_original.iloc[new_indices]
proba_rare = proba_original[new_indices]

# Recalcula AUC e AP com a classe positiva rara
from sklearn.metrics import roc_auc_score, average_precision_score

auc_rare = roc_auc_score(y_test_rare, proba_rare)
ap_rare = average_precision_score(y_test_rare, proba_rare)

print(f"AUC original: {auc:.4f}")
print(f"AP original: {ap:.4f}")
print("---")
print(f"AUC com positivo mais raro: {auc_rare:.4f}")
print(f"AP com positivo mais raro: {ap_rare:.4f}")

print("\nObservação: A AP deve cair mais significativamente que a AUC quando a classe positiva se torna mais rara, demonstrando sua maior sensibilidade a cenários de desbalanceamento severo.")

AUC original: 0.9147
AP original: 0.6698
---
AUC com positivo mais raro: 0.9500
AP com positivo mais raro: 0.6257

Observação: A AP deve cair mais significativamente que a AUC quando a classe positiva se torna mais rara, demonstrando sua maior sensibilidade a cenários de desbalanceamento severo.


## Part 4 — Log-loss: the confidence of probabilities
## Parte 4 — Log-loss: a confiança das probabilidades

**Exercise 10.** Calculate the **log-loss** of the model and that of a **baseline** that always predicts the positive rate. Is the model better than the baseline?
**Exercício 10.** Calcule o **log-loss** do modelo e o de uma **base** que prevê sempre a taxa de positivos. O modelo é melhor que a base?

In [ ]:
from sklearn.metrics import log_loss

# Log-loss do modelo
logloss_modelo = log_loss(y_test, proba)
print(f"Log-loss do Modelo: {logloss_modelo:.4f}")

# Log-loss da baseline (modelo que prevê sempre a taxa de positivos)
# A baseline prevê a mesma probabilidade (a taxa de positivos) para todas as instâncias.
# Usamos 'positive_rate' calculado anteriormente.

# Criamos um array de probabilidades com o valor da taxa de positivos para todos os exemplos
proba_baseline = np.full(len(y_test), positive_rate)
logloss_baseline = log_loss(y_test, proba_baseline)
print(f"Log-loss da Baseline (taxa de positivos): {logloss_baseline:.4f}")

# Comparação
if logloss_modelo < logloss_baseline:
    print("\nComparação: O Log-loss do modelo é menor que o da baseline, indicando que o modelo é melhor em atribuir probabilidades calibradas do que um modelo que chuta a média.")
else:
    print("\nComparação: O Log-loss do modelo é maior ou igual ao da baseline, o que pode indicar que o modelo não está superando um chute simples.")

Log-loss do Modelo: 0.2143
Log-loss da Baseline (taxa de positivos): 0.3682

Comparação: O Log-loss do modelo é menor que o da baseline, indicando que o modelo é melhor em atribuir probabilidades calibradas do que um modelo que chuta a média.


**Exercise 11 (confident error).** Compare the log-loss of ONE positive case predicted with p = 0.9 (confident and correct) and with p = 0.1 (confident and WRONG). See how confident error is penalized.
**Exercício 11 (erro confiante).** Compare o log-loss de UM caso positivo previsto com p = 0,9 (confiante e certo) e com p = 0,1 (confiante e ERRADO). Veja como o erro confiante é punido.

In [ ]:
from sklearn.metrics import log_loss
import numpy as np

# Caso 1: Positivo real (y_true = 1), previsto com alta confiança e CORRETAMENTE (p_pred = 0.9)
logloss_confiante_certo = log_loss([1], [0.9], labels=[0, 1])
print(f"Log-loss (y=1, p=0.9, confiante e CORRETO): {logloss_confiante_certo:.4f}")

# Caso 2: Positivo real (y_true = 1), previsto com alta confiança e ERRADAMENTE (p_pred = 0.1)
# Note que o log_loss penaliza a previsão 0.1 para um y_true de 1. O modelo está 'confiante' de que não é um hit.
logloss_confiante_errado = log_loss([1], [0.1], labels=[0, 1])
print(f"Log-loss (y=1, p=0.1, confiante e ERRADO): {logloss_confiante_errado:.4f}")

print("\nObservação: O Log-loss para o erro confiante (p=0.1 para um hit) é muito maior do que para o acerto confiante (p=0.9 para um hit). Isso demonstra como o log-loss penaliza severamente as previsões incorretas feitas com alta confiança.")

Log-loss (y=1, p=0.9, confiante e CORRETO): 0.1054
Log-loss (y=1, p=0.1, confiante e ERRADO): 2.3026

Observação: O Log-loss para o erro confiante (p=0.1 para um hit) é muito maior do que para o acerto confiante (p=0.9 para um hit). Isso demonstra como o log-loss penaliza severamente as previsões incorretas feitas com alta confiança.


**Exercise 12.** Consolidate everything into a single summary: AUC, AP, and log-loss of the model on the test set (a `print` statement with all three).
**Exercício 12.** Junte tudo num único resumo: AUC, AP e log-loss do modelo no teste (um `print` com os três).

In [ ]:
print(f"### Resumo das Métricas do Modelo ###")
print(f"AUC no Teste: {auc:.4f}")
print(f"Average Precision (AP) no Teste: {ap:.4f}")
print(f"Log-loss no Teste: {logloss_modelo:.4f}")

print("\nObservação: Esses valores fornecem uma visão abrangente do desempenho do modelo, considerando sua capacidade de distinguir classes (AUC), sua performance em cenários de classe rara (AP) e a calibração de suas probabilidades (Log-loss).")

### Resumo das Métricas do Modelo ###
AUC no Teste: 0.9147
Average Precision (AP) no Teste: 0.6698
Log-loss no Teste: 0.2143

Observação: Esses valores fornecem uma visão abrangente do desempenho do modelo, considerando sua capacidade de distinguir classes (AUC), sua performance em cenários de classe rara (AP) e a calibração de suas probabilidades (Log-loss).


## Part 5 — Choosing the threshold by objective
## Parte 5 — Escolher o limiar pelo objetivo

**Exercise 13.** Find the **highest** threshold that still guarantees **recall ≥ 0.80** (we want to catch 80% of the hits). Then, what is the precision at that threshold?
**Exercício 13.** Encontre o **maior** limiar que ainda garante **recall ≥ 0,80** (queremos pegar 80% dos hits). Depois, qual a precisão nesse limiar?

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

# Já temos 'precision', 'recall' e 'thresholds' calculados do Exercício 8

# Encontrar o maior limiar que garante recall >= 0.80
# Itera sobre os thresholds e encontra o primeiro (maior) que satisfaz a condição
# Os arrays de precision/recall/thresholds são ordenados de forma que thresholds é crescente
# Para encontrar o 'maior limiar', precisamos iterar de trás para frente ou encontrar o índice correto.
# É mais fácil encontrar os índices onde recall >= 0.80 e pegar o limiar correspondente ao maior desses índices.

# Encontra todos os índices onde o recall é maior ou igual a 0.80
indices_recall_ge_080 = np.where(recall >= 0.80)[0]

if len(indices_recall_ge_080) > 0:
    # O maior limiar que satisfaz a condição corresponderá ao índice mais alto na lista de thresholds
    # (ou seja, o menor índice no array 'precision_recall_curve' onde o recall ainda é >= 0.80)
    # precision_recall_curve retorna os thresholds em ordem crescente, mas o recall e precision são decrescentes
    # Precisamos encontrar o primeiro threshold (o maior) onde recall é >= 0.80

    # Encontrar o índice do primeiro ponto onde o recall é >= 0.80 (este será o maior limiar)
    # Os thresholds são associados ao ponto inicial do intervalo, então o último threshold é 'None'
    # Assim, o array 'thresholds' tem um elemento a menos que 'precision' e 'recall'.
    # Usaremos 'len(thresholds) -1' para o último ponto válido.

    # Encontra os índices onde o recall atende ao critério
    valid_indices = np.where(recall[:-1] >= 0.80)[0] # Exclui o último ponto de recall/precision que não tem threshold

    if len(valid_indices) > 0:
        # O maior limiar que garante recall >= 0.80 é o threshold correspondente ao menor índice do valid_indices
        # Isso ocorre porque os thresholds são ordenados de forma crescente e o recall de forma decrescente
        best_idx = valid_indices[0]
        limiar_desejado = thresholds[best_idx]
        precisao_no_limiar = precision[best_idx]

        print(f"Maior limiar para recall >= 0.80: {limiar_desejado:.4f}")
        print(f"Precisão neste limiar: {precisao_no_limiar:.4f}")
    else:
        print("Não foi encontrado nenhum limiar que garanta recall >= 0.80.")
else:
    print("Não foi encontrado nenhum limiar que garanta recall >= 0.80.")

print("\nObservação: Se o objetivo é não deixar hits passarem (alto recall), frequentemente se sacrifica a precisão, pois o modelo precisa ser mais permissivo, aumentando também os falsos positivos.")

Maior limiar para recall >= 0.80: 0.0000
Precisão neste limiar: 0.1206

Observação: Se o objetivo é não deixar hits passarem (alto recall), frequentemente se sacrifica a precisão, pois o modelo precisa ser mais permissivo, aumentando também os falsos positivos.


**Exercise 14 (asymmetric cost).** Suppose a False Negative costs 5 and a False Positive costs 1. Sweep thresholds from 0.05 to 0.95 and choose the one that **minimizes the total cost`.
**Exercício 14 (custo assimétrico).** Suponha que um Falso Negativo custa 5 e um Falso Positivo custa 1. Varra limiares de 0,05 a 0,95 e escolha o que **minimiza o custo total`.

In [ ]:
from sklearn.metrics import confusion_matrix

# Definir os custos
custo_falso_negativo = 5
custo_falso_positivo = 1

melhor_custo = float('inf')
melhor_limiar = None

print("Varrendo limiares e calculando o custo total:")
for limiar in np.arange(0.05, 1.0, 0.05):
    # Gera as previsões para o limiar atual
    y_pred_limiar = (proba >= limiar).astype(int)

    # Calcula a matriz de confusão
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_limiar).ravel()

    # Calcula o custo total
    custo_total = (fp * custo_falso_positivo) + (fn * custo_falso_negativo)

    print(f"  Limiar: {limiar:.2f} | FP: {fp}, FN: {fn} | Custo Total: {custo_total:.2f}")

    if custo_total < melhor_custo:
        melhor_custo = custo_total
        melhor_limiar = limiar

print(f"\nO limiar que minimiza o custo total é: {melhor_limiar:.2f} com um Custo Total de: {melhor_custo:.2f}")

print("\nObservação: A escolha do limiar ideal depende criticamente dos custos (ou benefícios) associados a cada tipo de erro e acerto, refletindo a realidade do problema de negócio.")

Varrendo limiares e calculando o custo total:
  Limiar: 0.05 | FP: 92, FN: 4 | Custo Total: 112.00
  Limiar: 0.10 | FP: 61, FN: 5 | Custo Total: 86.00
  Limiar: 0.15 | FP: 42, FN: 5 | Custo Total: 67.00
  Limiar: 0.20 | FP: 33, FN: 9 | Custo Total: 78.00
  Limiar: 0.25 | FP: 23, FN: 10 | Custo Total: 73.00
  Limiar: 0.30 | FP: 15, FN: 12 | Custo Total: 75.00
  Limiar: 0.35 | FP: 13, FN: 14 | Custo Total: 83.00
  Limiar: 0.40 | FP: 9, FN: 17 | Custo Total: 94.00
  Limiar: 0.45 | FP: 8, FN: 21 | Custo Total: 113.00
  Limiar: 0.50 | FP: 5, FN: 23 | Custo Total: 120.00
  Limiar: 0.55 | FP: 3, FN: 24 | Custo Total: 123.00
  Limiar: 0.60 | FP: 2, FN: 24 | Custo Total: 122.00
  Limiar: 0.65 | FP: 2, FN: 27 | Custo Total: 137.00
  Limiar: 0.70 | FP: 1, FN: 30 | Custo Total: 151.00
  Limiar: 0.75 | FP: 1, FN: 30 | Custo Total: 151.00
  Limiar: 0.80 | FP: 1, FN: 31 | Custo Total: 156.00
  Limiar: 0.85 | FP: 1, FN: 34 | Custo Total: 171.00
  Limiar: 0.90 | FP: 0, FN: 36 | Custo Total: 180.00
  Li

## Part 6 — Multiclass: macro, micro, and weighted averages
## Parte 6 — Multiclasse: médias macro, micro e weighted

**Exercise 15.** In the `patients` dataset, predict `risco_metabolico` (3 classes) with a LogisticRegression. Show the `classification_report` (precision/recall per class + averages).
**Exercício 15.** Na base `pacientes`, preveja `risco_metabolico` (3 classes) com uma LogisticRegression. Mostre o `classification_report` (precisão/recall por classe + as médias).

In [ ]:
from sklearn.metrics import classification_report

# Carregar a base de pacientes
pacientes = pd.read_csv('/content/1_pacientes.csv')

# Definir features (X) e target (y)
X_pacientes = pacientes[['idade', 'imc']]
y_pacientes = pacientes['risco_metabolico']

# Dividir em treino e teste
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pacientes, y_pacientes, test_size=0.3, random_state=42, stratify=y_pacientes)

# Treinar o modelo de Regressão Logística
modelo_multiclasse = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train_p, y_train_p)

# Fazer previsões
y_pred_multiclasse = modelo_multiclasse.predict(X_test_p)

# Exibir o classification_report
print(classification_report(y_test_p, y_pred_multiclasse))

              precision    recall  f1-score   support

        alto       0.56      0.13      0.21        38
       baixo       0.69      0.88      0.77       153
    moderado       0.46      0.38      0.41        64

    accuracy                           0.64       255
   macro avg       0.57      0.46      0.47       255
weighted avg       0.61      0.64      0.60       255



**Exercise 16.** Calculate the F1 **macro**, **micro**, and **weighted** and confirm that **micro == accuracy** (in single-label problems).
**Exercício 16.** Calcule o F1 **macro**, **micro** e **weighted** e confirme que o **micro == acurácia** (em problema de rótulo único).

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Calculate F1-score with different averaging methods
f1_macro = f1_score(y_test_p, y_pred_multiclasse, average='macro')
f1_micro = f1_score(y_test_p, y_pred_multiclasse, average='micro')
f1_weighted = f1_score(y_test_p, y_pred_multiclasse, average='weighted')

print(f"F1-score (Macro): {f1_macro:.4f}")
print(f"F1-score (Micro): {f1_micro:.4f}")
print(f"F1-score (Weighted): {f1_weighted:.4f}")

# Confirm that micro F1-score is equal to accuracy for single-label classification
accuracy = accuracy_score(y_test_p, y_pred_multiclasse)
print(f"Acurácia: {accuracy:.4f}")
print(f"F1-score (Micro) == Acurácia: {f1_micro:.4f} == {accuracy:.4f} -> {f1_micro == accuracy}")

F1-score (Macro): 0.4663
F1-score (Micro): 0.6392
F1-score (Weighted): 0.5990
Acurácia: 0.6392
F1-score (Micro) == Acurácia: 0.6392 == 0.6392 -> True


## Part 7 — Comparing models and synthesizing
## Parte 7 — Comparar modelos e sintetizar

**Exercise 17.** Compare your model with a **naive baseline** (`DummyClassifier` that predicts the probability of the majority class) using AUC, AP, and log-loss. Does your model win in all three?
**Exercício 17.** Compare o seu modelo com uma **base ingênua** (`DummyClassifier` que prevê a probabilidade da classe majoritária) por AUC, AP e log-loss. O seu modelo ganha nas três?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

# 1. Treinar um DummyClassifier (baseline ingênua)
# 'stratified' prevê probabilidades baseadas na distribuição de classes de treino.
# 'most_frequent' prevê sempre a classe majoritária.
# Para a comparação com AUC, AP e Log-loss, que usam probabilidades, 'stratified' é mais apropriado
# como baseline que gera probabilidades (ainda que aleatórias dentro da distribuição).
# No entanto, o enunciado pede "prevê a probabilidade da classe majoritária", o que é mais próximo de 'most_frequent'
# se interpretarmos que a probabilidade da classe majoritária é 1 para a classe majoritária.
# Se a intenção é ter uma probabilidade constante igual à taxa da classe majoritária, precisaríamos fazer isso manualmente.
# Como o pedido é para "prevê a probabilidade da classe majoritária", vamos usar 'most_frequent' e para o proba, assumir a taxa de positivos.

dummy_clf = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_clf.fit(X_train, y_train)

# Para calcular AUC, AP e Log-loss, precisamos de probabilidades.
# O DummyClassifier 'most_frequent' não gera probabilidades de forma padrão.
# Uma forma de obter uma baseline para probabilidades é usar a taxa de positivos como a probabilidade constante.
# Ou, alternativamente, usar strategy='prior' que prevê probabilidades baseadas na distribuição das classes.
# Vamos usar a taxa de positivos como probabilidade constante para a baseline, pois isso é uma "previsão da classe majoritária" de forma probabilística.

# Taxa de positivos no conjunto de treino (usado como baseline constante)
proba_dummy = np.full_like(y_test, y_train.mean(), dtype=float)

# 2. Calcular as métricas para o DummyClassifier
auc_dummy = roc_auc_score(y_test, proba_dummy)
ap_dummy = average_precision_score(y_test, proba_dummy)
logloss_dummy = log_loss(y_test, proba_dummy)

# 3. Comparar com o modelo principal (valores já calculados: auc, ap, logloss_modelo)
print(f"### Comparação: Modelo Principal vs. Baseline Ingênua (DummyClassifier) ###\n")

print(f"--- Modelo Principal ---")
print(f"  AUC: {auc:.4f}")
print(f"  AP: {ap:.4f}")
print(f"  Log-loss: {logloss_modelo:.4f}\n")

print(f"--- Baseline Ingênua (DummyClassifier - Prevê taxa de positivos) ---")
print(f"  AUC: {auc_dummy:.4f}")
print(f"  AP: {ap_dummy:.4f}")
print(f"  Log-loss: {logloss_dummy:.4f}\n")

# Verificar se o modelo principal ganha nas três métricas
modelo_ganha_auc = auc > auc_dummy
modelo_ganha_ap = ap > ap_dummy
modelo_ganha_logloss = logloss_modelo < logloss_dummy # Menor log-loss é melhor

if modelo_ganha_auc and modelo_ganha_ap and modelo_ganha_logloss:
    print("Conclusão: O modelo principal ganha em todas as três métricas (AUC, AP e Log-loss) comparado à baseline ingênua.")
else:
    print("Conclusão: O modelo principal NÃO ganha em todas as três métricas comparado à baseline ingênua.")


### Comparação: Modelo Principal vs. Baseline Ingênua (DummyClassifier) ###

--- Modelo Principal ---
  AUC: 0.9147
  AP: 0.6698
  Log-loss: 0.2143

--- Baseline Ingênua (DummyClassifier - Prevê taxa de positivos) ---
  AUC: 0.5000
  AP: 0.1206
  Log-loss: 0.3682

Conclusão: O modelo principal ganha em todas as três métricas (AUC, AP e Log-loss) comparado à baseline ingênua.


Comparison: Main Model vs. Naive Baseline (DummyClassifier)
--- Main Model ---
  AUC: 0.9147
  AP: 0.6698
  Log-loss: 0.2143

--- Naive Baseline (DummyClassifier - Predicts positive rate) ---
  AUC: 0.5000
  AP: 0.1206
  Log-loss: 0.3682

Conclusion: The main model wins in all three metrics (AUC, AP, and Log-loss) compared to the naive baseline.


Comparação: Modelo Principal vs. Baseline Ingênua (DummyClassifier)
--- Modelo Principal --- AUC: 0.9147 AP: 0.6698 Log-loss: 0.2143

--- Baseline Ingênua (DummyClassifier - Prevê taxa de positivos) --- AUC: 0.5000 AP: 0.1206 Log-loss: 0.3682

Conclusão: O modelo principal ganha em todas as três métricas (AUC, AP e Log-loss) comparado à baseline ingênua.

**Exercise 18 (synthesis).** Write **three conclusions**: (a) why, for predicting `hit` (rare), **AP** tells a different story than **AUC**; (b) what **log-loss** adds that AUC does not see; (c) how you would choose the **threshold** if the goal was to not miss any hits.
**Exercício 18 (síntese).** Escreva **três conclusões**: (a) por que, para prever `hit` (raro), a **AP** conta uma história diferente da **AUC**; (b) o que o **log-loss** acrescenta que a AUC não vê; (c) como você escolheria o **limiar** se o objetivo fosse não deixar hits passarem.

### Exercise 18 (Synthesis) - Detailed Answers

Let's explore in more detail the conclusions about our model's metrics, as requested!

### Exercício 18 (síntese) - Respostas Detalhadas

Vamos explorar em mais detalhes as conclusões sobre as métricas do nosso modelo, conforme solicitado!

### **(a) Why, for predicting `hit` (which is rare), does AP tell a different story than AUC?**

Think of it this way: **AUC (Area Under the Receiver Operating Characteristic curve)** is like a 'general judge' of the model. It looks at how well the model can differentiate positive from negative classes in *all* situations, without paying much attention to class imbalance. It cares more about the order of predictions: if the model gives a higher score to a 'hit' than to a 'non-hit', that's good for AUC.

**AP (Average Precision)**, on the other hand, is more like a 'specialized detective' for rare classes. It focuses on finding true 'hits' when they are few. If the model starts generating many 'false alarms' (false positives) while trying to find all 'hits', AP drops drastically. This is because it evaluates Precision (how many of those the model said were 'hits' actually were) in relation to Recall (how many true 'hits' the model managed to catch).

**In summary:** When what you want to predict is rare, AP gives you a much more realistic view of how useful your model is in practice, because it penalizes more the errors that truly matter in this scenario. AUC might look great, but AP shows if your model is *precise* when identifying those few important cases.

### **(a) Por que, para prever `hit` (que é algo raro), a AP conta uma história diferente da AUC?**

Pense assim: a **AUC (Area Under the Receiver Operating Characteristic curve)** é como um 'juiz geral' do modelo. Ela olha para quão bem o modelo consegue diferenciar as classes positivas das negativas em *todas* as situações, sem dar muita atenção ao desbalanceamento de classes. Ela se preocupa mais com a ordem das previsões: se o modelo dá uma nota mais alta para um 'hit' do que para um 'não-hit', isso é bom para a AUC.

Já a **AP (Average Precision)** é mais como um 'detetive especializado' para classes raras. Ela se foca em encontrar os 'hits' de verdade quando eles são poucos. Se o modelo começa a gerar muitos 'alarmes falsos' (falsos positivos) ao tentar achar todos os 'hits', a AP cai drasticamente. Isso porque ela avalia a Precisão (quantos dos que o modelo disse que eram 'hits' realmente eram) em relação ao Recall (quantos 'hits' de verdade o modelo conseguiu pegar).

**Em resumo:** Quando o que você quer prever é raro, a AP te dá uma visão muito mais realista de quão útil seu modelo é na prática, porque ela penaliza mais os erros que realmente importam nesse cenário. A AUC pode parecer ótima, mas a AP mostra se seu modelo é *preciso* na hora de identificar aqueles poucos casos importantes.

### **(b) What does `log-loss` add that AUC doesn't see?**

Let's think of **AUC** as an evaluator who only cares about the order of arrival. It wants to know if your model placed 'hits' ahead of 'non-hits'. If the model gave a probability of 0.9 for a 'hit' and 0.6 for another 'hit', and 0.2 for a 'non-hit', AUC is happy because the order is correct.

But what if the model was *very confident* in an error? Like, it said with 99% certainty that something was a 'hit', but it was actually a 'non-hit'? AUC might not care much, as long as the general order is good.

That's where **Log-loss** comes in! It's a 'confidence inspector'. Log-loss cares *a lot* about how **confident** your model is in its predictions. It severely punishes incorrect predictions made with *high confidence*. If your model says something is a 'hit' with 90% certainty, and it really is, the Log-loss is low. But if it says with 90% certainty that it's a 'hit', and it's not, the Log-loss skyrockets!

**Conclusion:** While AUC tells you if the model can rank well, Log-loss tells you if the probabilities the model provides are *reliable* and *well-calibrated*. In situations where prediction 'certainty' is crucial (such as in medical diagnoses or financial risks), Log-loss is indispensable.

### **(b) O que o `log-loss` acrescenta que a AUC não vê?**

Vamos pensar que a **AUC** é como um avaliador que só se importa com a ordem de chegada. Ele quer saber se seu modelo colocou os 'hits' na frente dos 'não-hits'. Se o modelo deu uma probabilidade de 0.9 para um 'hit' e 0.6 para outro 'hit', e 0.2 para um 'não-hit', a AUC está feliz porque a ordem está correta.

Mas e se o modelo estivesse *muito confiante* num erro? Tipo, disse com 99% de certeza que algo era um 'hit', mas era um 'não-hit' na verdade? A AUC pode não se importar tanto, desde que a ordem geral esteja boa.

É aí que entra o **Log-loss**! Ele é um 'fiscal de confiança'. O Log-loss se importa *muito* com o quanto o seu modelo está **confiante** em suas previsões. Ele castiga com rigor as previsões erradas feitas com *alta confiança*. Se seu modelo diz que algo é um 'hit' com 90% de certeza, e realmente é, o Log-loss é baixo. Mas se ele diz com 90% de certeza que é um 'hit', e não é, o Log-loss dispara!

**Conclusão:** Enquanto a AUC te diz se o modelo consegue ranquear bem, o Log-loss te informa se as probabilidades que o modelo dá são *confiáveis* e *bem calibradas*. Em situações onde a 'certeza' da previsão é crucial (como em diagnósticos médicos ou riscos financeiros), o Log-loss é indispensável.

### **(c) How would you choose the `threshold` if the goal was to not miss any `hits`?**

When the main goal is **to not let any 'hit' escape**, what we are looking for is to maximize **Recall** (also known as sensitivity). Recall tells us the proportion of actual 'hits' that our model managed to identify.

For this, I would do the following:

1.  **Look at the Precision-Recall curve:** This curve shows us the relationship between Precision and Recall at different cutoff points (thresholds) of the model.
2.  **Find the desired Recall:** My focus would be to identify the thresholds that give me a high Recall, say, 80%, 90% or even more, depending on the critical need not to miss any 'hit'.
3.  **Choose the 'highest' threshold:** Within this high Recall range, I would choose the *highest* possible threshold. Why? Because a higher threshold tends to be a bit more conservative, and generally results in slightly better Precision, minimizing (but not eliminating) false positives, without compromising the main objective of Recall.

In Exercise 13, we saw that a very low threshold (close to zero) gave us the highest Recall, but with low Precision. This is the 'trade-off' we accept when the most important thing is to 'catch everything', even if it means some extra false alarms. The goal is 'total coverage' of 'hits'!

### **(c) Como você escolheria o `limiar` se o objetivo fosse não deixar `hits` passarem?**

Quando o objetivo principal é **não deixar nenhum 'hit' escapar**, o que estamos buscando é maximizar o **Recall** (também conhecido como sensibilidade). O Recall nos diz a proporção de 'hits' reais que nosso modelo conseguiu identificar.

Para isso, eu faria o seguinte:

1.  **Olharia para a curva de Precisão-Recall:** Essa curva nos mostra a relação entre a Precisão e o Recall em diferentes pontos de corte (limiares) do modelo.
2.  **Encontraria o Recall desejado:** Meu foco seria identificar os limiares que me dão um Recall alto, digamos, de 80%, 90% ou até mais, dependendo da necessidade crítica de não perder nenhum 'hit'.
3.  **Escolheria o 'maior' limiar:** Dentro dessa faixa de Recall alto, eu escolheria o *maior* limiar possível. Por quê? Porque um limiar maior tende a ser um pouco mais conservador, e geralmente resulta em uma Precisão um pouco melhor, minimizando (mas não eliminando) os falsos positivos, sem comprometer o objetivo principal de Recall.

No Exercício 13, vimos que um limiar muito baixo (próximo de zero) nos deu o maior Recall, mas com uma Precisão baixa. Essa é a 'troca' que aceitamos quando o mais importante é 'pegar tudo', mesmo que signifique alguns falsos alarmes extras. O objetivo é a 'cobertura total' dos 'hits'!